# Bird Aero Vibration POC: Structural Deformation & Vibration Analysis with MeshGraphNet

This notebook demonstrates a complete data-driven pipeline for **structural deformation prediction and vibration analysis** using **MeshGraphNet**, a graph neural network architecture originally developed by DeepMind.

**Use case:** [Bird Aero](https://www.birdaero.com/) develops missile protection pods (AEROSHIELD, AMPS), DIRCM systems (SPREOS), and surveillance payloads (ASIO, OSCAR) mounted on military and VIP aircraft. These structures are subject to complex vibration loads from engine harmonics and rotor-induced excitation. Traditional FEA simulation and physical vibration testing are accurate but slow and expensive. This POC demonstrates that a GNN surrogate model trained on FEA data can:

1. **Predict structural deformation** orders of magnitude faster than full FEA
2. **Perform autoregressive rollout** — forecast multi-step deformation trajectories
3. **Extract vibration characteristics** — identify dominant frequencies and mode shapes via FFT

**Dataset:** DeepMind's open-source *deforming plate* dataset — 1,000 COMSOL FEA simulations of a plate under varying loads, ~1,271 mesh nodes, 400 timesteps each.

**Pipeline:**
```
Download FEA data → Build graph representations → Train MeshGraphNet
    → Autoregressive rollout → FFT frequency analysis → Vibration mode shapes
```

## 1. Setup

Clone the repository and install dependencies. This cell is designed for Google Colab.

In [ ]:
# Clone repo and install
!git clone https://github.com/orireshef/birdaero-vibration-poc.git
%cd birdaero-vibration-poc
!pip install . -q

import sys
sys.path.insert(0, "src")

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Using device: {device}")

## 2. Download Data

Download the DeepMind deforming plate dataset from Google Cloud Storage. This includes `meta.json` and TFRecord files for train/valid/test splits.

In [ ]:
%%time
from vibration_poc.dataset.config import DatasetConfig
from vibration_poc.dataset.download import download_dataset

config = DatasetConfig()
download_dataset(config)
print("Download complete.")
print(f"Raw data directory: {config.raw_dir}")
!ls -lh {config.raw_dir}

## 3. Preprocess

Convert raw TFRecords into graph representations:
- **Nodes** = mesh vertices (features: world position + node type)
- **Edges** = mesh connectivity from tetrahedral cells (features: relative position + distance)
- **Target** = next-step displacement (world_pos_{t+1} - world_pos_t)

Normalization stats are computed on the training split and applied to all splits.

In [ ]:
%%time
from vibration_poc.dataset.preprocess import preprocess_dataset

preprocess_dataset(config)
print("Preprocessing complete.")

# Print stats: number of graphs per split
from pathlib import Path
for split in config.splits:
    split_dir = config.processed_dir / split
    n_graphs = len(list(split_dir.glob("*.pt")))
    print(f"  {split}: {n_graphs} graphs")

## 4. Inspect Data

Load a single graph and examine its structure. Each graph represents one time-step transition in the FEA simulation.

In [ ]:
from vibration_poc.dataset.dataloader import GraphDataset

ds = GraphDataset(config.processed_dir, "train")
print(f"Training dataset size: {len(ds)} graphs\n")

sample = ds[0]
print("Graph tensors:")
for key, val in sample.items():
    print(f"  {key:20s} shape={str(list(val.shape)):15s} dtype={val.dtype}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Visualize the mesh as a 3D scatter plot
mesh_pos = sample["mesh_pos"].numpy()
y = sample["y"].numpy()
disp_mag = np.linalg.norm(y, axis=1)

fig = plt.figure(figsize=(12, 5))

# Mesh colored by displacement magnitude
ax = fig.add_subplot(121, projection="3d")
sc = ax.scatter(
    mesh_pos[:, 0], mesh_pos[:, 1], mesh_pos[:, 2],
    c=disp_mag, cmap="hot", s=2, alpha=0.8
)
ax.set_title("Mesh (colored by displacement magnitude)")
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")
plt.colorbar(sc, ax=ax, shrink=0.6, label="|displacement|")

# Node type distribution
ax2 = fig.add_subplot(122)
node_types = sample["x"][:, 3].numpy()  # last column is node_type (normalized)
ax2.hist(node_types, bins=30, edgecolor="black")
ax2.set_title("Node Type Distribution (normalized)")
ax2.set_xlabel("Node type value")
ax2.set_ylabel("Count")

plt.tight_layout()
plt.show()

print(f"\nMesh stats:")
print(f"  Nodes: {mesh_pos.shape[0]}")
print(f"  Edges: {sample['edge_index'].shape[1]}")
print(f"  Displacement range: [{disp_mag.min():.6f}, {disp_mag.max():.6f}]")

## 5. Create Model

Instantiate MeshGraphNet with POC configuration: hidden_dim=64, num_layers=8.

Architecture: **Encoder** (node/edge MLPs) → **Processor** (8 message-passing layers with residual connections) → **Decoder** (output MLP predicting 3D displacement).

In [ ]:
from vibration_poc.model.meshgraphnet import MeshGraphNet

model = MeshGraphNet(
    input_dim_nodes=4,   # world_pos (3) + node_type (1)
    input_dim_edges=4,   # relative_pos (3) + distance (1)
    output_dim=3,        # displacement (dx, dy, dz)
    hidden_dim=64,
    num_layers=8,
)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"\nModel architecture:")
print(model)

## 6. Train

Train for 5 epochs using the POC config. On Colab with a T4 GPU this should complete in a few minutes.

- **Loss**: MSE on predicted displacement (next-step velocity)
- **Optimizer**: Adam, LR=1e-4, exponential decay
- **Training**: 1-step prediction only (no multi-step rollout loss)

In [ ]:
%%time
from vibration_poc.training.trainer import TrainingConfig, train

train_config = TrainingConfig(
    epochs=5,
    learning_rate=1e-4,
    lr_decay=0.9999991,
    hidden_dim=64,
    num_layers=8,
    batch_size=1,
    device=device,
    checkpoint_dir="checkpoints",
    log_interval=1,
)

best_checkpoint = train(train_config, config)
print(f"\nBest checkpoint saved to: {best_checkpoint}")

## 7. Physics-Aware Training

The baseline model above minimizes pure MSE on displacement predictions. But a structural surrogate should also respect **physical constraints** from the original FEA simulation:

1. **Boundary condition enforcement** — Fixed/clamped nodes (types 1, 2, 3) should have near-zero displacement. The FEA solver enforces this exactly; our surrogate should too.
2. **Stress consistency** — When the model predicts stress as a secondary output, it should match the FEA-computed stress field. This dual-output training acts as a physics-informed regularizer.
3. **Smoothness** — Neighboring nodes in a continuous material should deform similarly. A smoothness penalty on displacement gradients enforces material continuity.

These constraints ensure the surrogate model respects the same physical laws as the original FEA simulation — critical for Bird Aero, where structural predictions drive safety-critical design decisions.

In [ ]:
%%time
from vibration_poc.physics import PhysicsConfig

physics_config = PhysicsConfig(
    bc_node_types=[1, 2, 3],
    bc_loss_weight=1.0,
    stress_loss_weight=0.1,
    smoothness_weight=0.01,
)

physics_train_config = TrainingConfig(
    epochs=5,
    learning_rate=1e-4,
    lr_decay=0.9999991,
    hidden_dim=64,
    num_layers=8,
    batch_size=1,
    device=device,
    checkpoint_dir="checkpoints_physics",
    log_interval=1,
    physics=physics_config,
)

physics_checkpoint = train(physics_train_config, config)
print(f"Physics-aware checkpoint: {physics_checkpoint}")

In [ ]:
physics_model = MeshGraphNet(
    input_dim_nodes=4, input_dim_edges=4, output_dim=3,
    hidden_dim=64, num_layers=8, predict_stress=True,
)
physics_model.load_state_dict(torch.load(physics_checkpoint, weights_only=True))
physics_model = physics_model.to(eval_device)
physics_model.eval()
print("Physics-aware model loaded")

## 8. Single-Step Evaluation

Load the best checkpoint and compute MSE on individual test samples — this measures how well the model predicts a single next-step displacement.

In [ ]:
import torch.nn.functional as F

# Load best checkpoint
eval_model = MeshGraphNet(
    input_dim_nodes=4,
    input_dim_edges=4,
    output_dim=3,
    hidden_dim=64,
    num_layers=8,
)
eval_model.load_state_dict(torch.load(best_checkpoint, weights_only=True))
eval_device = torch.device(device)
eval_model = eval_model.to(eval_device)
eval_model.eval()

# Evaluate on test set
test_ds = GraphDataset(config.processed_dir, "test")
print(f"Test set size: {len(test_ds)} graphs")

losses = []
num_eval = min(len(test_ds), 50)
with torch.no_grad():
    for i in range(num_eval):
        graph = {k: v.to(eval_device) for k, v in test_ds[i].items()}
        pred = eval_model(graph)
        loss = F.mse_loss(pred, graph["y"])
        losses.append(loss.item())

mean_mse = np.mean(losses)
std_mse = np.std(losses)
print(f"\nTest MSE ({num_eval} samples):")
print(f"  Mean: {mean_mse:.6f}")
print(f"  Std:  {std_mse:.6f}")
print(f"  Min:  {min(losses):.6f}")
print(f"  Max:  {max(losses):.6f}")

---

# Vibration Analysis Pipeline

The sections above train a single-step predictor. Below we use that model as a **surrogate simulator**: starting from an initial structural state, we roll forward in time autoregressively, then apply signal processing to extract vibration characteristics — exactly what Bird Aero needs for rapid design evaluation.

```
Trained model + initial state
    → Autoregressive rollout (multi-step trajectory)
    → Displacement time series per node
    → FFT → Dominant frequencies + mode shapes
```

## 9. Autoregressive Rollout

Feed predictions back as inputs to forecast a **multi-step deformation trajectory**. At each step:
1. Normalize current graph features using training statistics
2. Forward pass → predicted displacement
3. Update node positions: `world_pos += predicted_displacement`
4. Repeat

Edge features (mesh connectivity) remain static since the mesh topology doesn't change — only node positions evolve.

In [ ]:
%%time
import json
from vibration_poc.dataset.config import NormStats
from vibration_poc.inference.predict import rollout

# Load normalization stats from preprocessing
norm_stats_path = config.processed_dir / "norm_stats.json"
with open(norm_stats_path) as f:
    norm_stats = NormStats(**json.load(f))

# Use first test graph as initial state
initial_graph = test_ds[0]

# Rollout 50 steps into the future
NUM_ROLLOUT_STEPS = 50

# Baseline rollout (no BC enforcement)
results = rollout(
    eval_model,
    initial_graph,
    num_steps=NUM_ROLLOUT_STEPS,
    norm_stats=norm_stats,
    device=eval_device,
)

# Physics-aware rollout (with BC enforcement)
results_physics = rollout(
    physics_model,
    initial_graph,
    num_steps=NUM_ROLLOUT_STEPS,
    norm_stats=norm_stats,
    device=eval_device,
    bc_node_types=[1, 2, 3],
)

print(f"Rollout complete: {len(results)} steps (baseline), {len(results_physics)} steps (physics)")
print(f"World position shape: {results[0]['world_pos'].shape}")
print(f"Displacement shape:   {results[0]['predicted_displacement'].shape}")

# Track displacement evolution for both models
disp_per_step = [r["predicted_displacement"].cpu().numpy() for r in results]
mag_per_step = [np.linalg.norm(d, axis=1).mean() for d in disp_per_step]

disp_per_step_physics = [r["predicted_displacement"].cpu().numpy() for r in results_physics]
mag_per_step_physics = [np.linalg.norm(d, axis=1).mean() for d in disp_per_step_physics]

print(f"\nMean displacement magnitude over time:")
print(f"  {'Step':>8}  {'Baseline':>10}  {'Physics':>10}")
for step in [0, NUM_ROLLOUT_STEPS // 2, NUM_ROLLOUT_STEPS - 1]:
    print(f"  {step:>8}  {mag_per_step[step]:>10.6f}  {mag_per_step_physics[step]:>10.6f}")

In [ ]:
# Plot displacement magnitude evolution — baseline vs physics-aware
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: overall displacement comparison
axes[0].plot(mag_per_step, linewidth=2, label="Baseline", color="steelblue")
axes[0].plot(mag_per_step_physics, linewidth=2, label="Physics-aware", color="darkorange")
axes[0].set_xlabel("Rollout Step")
axes[0].set_ylabel("Mean Displacement Magnitude")
axes[0].set_title("Structural Response: Baseline vs Physics-Aware")
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Right: boundary node displacement (should be ~0 with physics)
node_types = initial_graph["x"][:, 3].numpy()
# BC nodes are types 1,2,3 — in normalized space, find them
# Use raw node type info: typically non-zero node_type values are boundary
bc_mask = node_types > 0  # boundary nodes have non-zero type
interior_mask = ~bc_mask

bc_mag_baseline = [np.linalg.norm(d[bc_mask], axis=1).mean() for d in disp_per_step]
bc_mag_physics = [np.linalg.norm(d[bc_mask], axis=1).mean() for d in disp_per_step_physics]

axes[1].plot(bc_mag_baseline, linewidth=2, label="Baseline (BC nodes)", color="steelblue", linestyle="--")
axes[1].plot(bc_mag_physics, linewidth=2, label="Physics-aware (BC nodes)", color="darkorange", linestyle="--")
axes[1].axhline(y=0, color="gray", linestyle=":", alpha=0.5)
axes[1].set_xlabel("Rollout Step")
axes[1].set_ylabel("Mean Displacement Magnitude")
axes[1].set_title("Boundary Node Displacement (should be ~0 with physics)")
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

if bc_mask.sum() > 0:
    print(f"\nBoundary nodes: {bc_mask.sum()} / {len(bc_mask)}")
    print(f"  Baseline mean BC displacement:      {np.mean(bc_mag_baseline):.6f}")
    print(f"  Physics-aware mean BC displacement:  {np.mean(bc_mag_physics):.6f}")
    reduction = (1 - np.mean(bc_mag_physics) / (np.mean(bc_mag_baseline) + 1e-8)) * 100
    print(f"  BC displacement reduction:           {reduction:.1f}%")

## 10. Deformation Visualization

Visualize the predicted structural deformation at key timesteps and as an animated GIF. This is analogous to what a Bird Aero engineer would see in a post-processing tool after running a full FEA simulation — but generated in seconds instead of hours.

In [ ]:
from vibration_poc.visualization.deformation import plot_deformation_snapshot

# Snapshots at key timesteps
timesteps = [0, NUM_ROLLOUT_STEPS // 4, NUM_ROLLOUT_STEPS // 2, NUM_ROLLOUT_STEPS - 1]
fig = plt.figure(figsize=(20, 5))

for idx, t in enumerate(timesteps):
    mesh = results[t]["mesh_pos"].cpu().numpy()
    disp = results[t]["predicted_displacement"].cpu().numpy()
    mag = np.linalg.norm(disp, axis=1)

    ax = fig.add_subplot(1, 4, idx + 1, projection="3d")
    sc = ax.scatter(
        mesh[:, 0], mesh[:, 1], mesh[:, 2],
        c=mag, cmap="hot", s=3
    )
    ax.set_title(f"Step {t}")
    fig.colorbar(sc, ax=ax, shrink=0.5, label="|disp|")

plt.suptitle("Predicted Deformation at Key Timesteps", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
from vibration_poc.visualization.deformation import create_deformation_gif

# Create animated deformation GIF
gif_path = Path("outputs/deformation.gif")
gif_path.parent.mkdir(parents=True, exist_ok=True)
create_deformation_gif(results, gif_path, fps=10)
print(f"Deformation GIF saved to: {gif_path}")

# Display in notebook (Colab)
from IPython.display import Image, display
display(Image(filename=str(gif_path)))

## 11. Prediction Error Analysis

Compare the model's first-step prediction against the ground truth from the FEA simulation. The 3-panel view shows: ground truth displacement magnitude, predicted displacement magnitude, and the absolute error — a standard validation approach in structural analysis.

In [ ]:
from vibration_poc.visualization.error_maps import plot_error_map

# First step: compare predicted vs ground truth
first_step = results[0]
mesh_np = first_step["mesh_pos"].cpu().numpy()
pred_np = first_step["predicted_displacement"].cpu().numpy()
gt_np = initial_graph["y"].numpy()  # ground truth from dataset

# Save error map
error_path = Path("outputs/error_map.png")
plot_error_map(mesh_np, gt_np, pred_np, save_path=error_path)

# Also display inline with matplotlib
gt_mag = np.linalg.norm(gt_np, axis=1)
pred_mag = np.linalg.norm(pred_np, axis=1)
error_mag = np.linalg.norm(gt_np - pred_np, axis=1)

vmin = min(gt_mag.min(), pred_mag.min())
vmax = max(gt_mag.max(), pred_mag.max())

fig = plt.figure(figsize=(18, 5))
for i, (title, vals, lo, hi, cmap) in enumerate([
    ("Ground Truth (FEA)", gt_mag, vmin, vmax, "hot"),
    ("Predicted (MeshGraphNet)", pred_mag, vmin, vmax, "hot"),
    ("Absolute Error", error_mag, None, None, "coolwarm"),
]):
    ax = fig.add_subplot(1, 3, i + 1, projection="3d")
    sc = ax.scatter(
        mesh_np[:, 0], mesh_np[:, 1], mesh_np[:, 2],
        c=vals, cmap=cmap, s=3, vmin=lo, vmax=hi,
    )
    ax.set_title(title)
    fig.colorbar(sc, ax=ax, shrink=0.5)

plt.suptitle("Single-Step Prediction: FEA Ground Truth vs MeshGraphNet", fontsize=14)
plt.tight_layout()
plt.show()

print(f"\nError statistics:")
print(f"  Mean absolute error: {error_mag.mean():.6f}")
print(f"  Max absolute error:  {error_mag.max():.6f}")
print(f"  Relative error:      {error_mag.mean() / (gt_mag.mean() + 1e-8):.4%}")

## 12. Frequency Analysis (FFT)

This is the key section for Bird Aero's vibration analysis use case. We extract displacement time series from the rollout, apply FFT, and identify **dominant vibration frequencies** — the same analysis an engineer would perform on accelerometer data or FEA output to check for resonance with engine/rotor harmonics.

In production, this enables:
- **Resonance avoidance**: verify that pod/payload natural frequencies don't coincide with aircraft excitation frequencies
- **Design iteration**: quickly evaluate how geometry changes affect vibration characteristics
- **Fatigue assessment**: identify high-amplitude frequency components that drive fatigue failure

In [ ]:
from vibration_poc.inference.analyze import (
    compute_fft,
    displacement_time_series,
    dominant_frequencies,
)

# Extract displacement magnitude time series [T, N]
series = displacement_time_series(results)
print(f"Displacement time series shape: {series.shape}  (steps x nodes)")

# Compute FFT
dt = 1.0  # normalized timestep
freqs, magnitudes = compute_fft(series, dt=dt)
print(f"FFT frequencies shape:  {freqs.shape}")
print(f"FFT magnitudes shape:   {magnitudes.shape}")

# Find dominant frequencies
top_freqs = dominant_frequencies(freqs, magnitudes, top_k=5)
print(f"\nDominant vibration frequencies:")
print(f"  {'Rank':>4}  {'Frequency':>12}  {'Mean Magnitude':>15}")
print(f"  {'----':>4}  {'----------':>12}  {'--------------':>15}")
for i, (freq, mag) in enumerate(top_freqs):
    print(f"  {i+1:>4}  {freq:>12.4f}  {mag:>15.6f}")

In [ ]:
from vibration_poc.visualization.frequency_analysis import plot_frequency_spectrum

# Plot frequency spectrum with dominant peaks marked
spectrum_path = Path("outputs/frequency_spectrum.png")
plot_frequency_spectrum(freqs, magnitudes, top_k=5, save_path=spectrum_path)

# Also display inline
mean_mag = magnitudes.mean(axis=1)
peak_indices = np.argsort(mean_mag)[-5:]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(freqs, mean_mag, linewidth=2, color="steelblue", label="Mean FFT magnitude")
ax.fill_between(freqs, mean_mag, alpha=0.2, color="steelblue")

for idx in peak_indices:
    ax.axvline(freqs[idx], color="red", linestyle="--", alpha=0.5)
    ax.plot(freqs[idx], mean_mag[idx], "rv", markersize=10)
    ax.annotate(
        f"f={freqs[idx]:.3f}",
        (freqs[idx], mean_mag[idx]),
        textcoords="offset points", xytext=(5, 10),
        fontsize=9, color="red",
    )

ax.set_xlabel("Frequency (normalized)", fontsize=12)
ax.set_ylabel("Mean FFT Magnitude", fontsize=12)
ax.set_title("Vibration Frequency Spectrum — Predicted Structural Response", fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 13. Vibration Mode Shapes

For each dominant frequency, visualize which parts of the structure vibrate most — the **mode shape**. Each subplot shows the mesh colored by per-node FFT magnitude at that frequency.

In practice, engineers compare these mode shapes against known excitation patterns (e.g., rotor blade-pass frequency) to identify structural vulnerabilities. A mode shape that concentrates stress near a mounting bracket, for example, signals a potential fatigue failure point.

In [ ]:
from vibration_poc.visualization.frequency_analysis import plot_mode_shapes

# Save mode shapes
modes_path = Path("outputs/mode_shapes.png")
plot_mode_shapes(mesh_np, magnitudes, freqs, top_k=3, save_path=modes_path)

# Display inline with enhanced formatting
top_k = 3
peak_indices_sorted = np.argsort(mean_mag)[-top_k:][::-1]

fig = plt.figure(figsize=(6 * top_k, 6))
for i, idx in enumerate(peak_indices_sorted):
    ax = fig.add_subplot(1, top_k, i + 1, projection="3d")
    node_mag = magnitudes[idx, :]
    sc = ax.scatter(
        mesh_np[:, 0], mesh_np[:, 1], mesh_np[:, 2],
        c=node_mag, cmap="hot", s=5,
    )
    fig.colorbar(sc, ax=ax, shrink=0.5, label="FFT magnitude")
    ax.set_title(f"Mode {i+1}: f = {freqs[idx]:.4f}", fontsize=13)
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")

plt.suptitle("Vibration Mode Shapes — Dominant Structural Resonances", fontsize=15)
plt.tight_layout()
plt.show()

## 14. Per-Node Displacement Spectrogram

Visualize how displacement evolves across all nodes over time — a heatmap where each row is a node and each column is a rollout timestep. This reveals spatial patterns in the structural response.

In [ ]:
# Displacement heatmap [T, N]
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Time-domain heatmap
im1 = axes[0].imshow(series.T, aspect="auto", cmap="viridis", interpolation="nearest")
axes[0].set_xlabel("Rollout Step")
axes[0].set_ylabel("Node Index")
axes[0].set_title("Displacement Magnitude Over Time")
fig.colorbar(im1, ax=axes[0], label="|displacement|")

# Frequency-domain heatmap
im2 = axes[1].imshow(magnitudes.T, aspect="auto", cmap="magma", interpolation="nearest")
axes[1].set_xlabel("Frequency Bin")
axes[1].set_ylabel("Node Index")
axes[1].set_title("FFT Magnitude per Node")
fig.colorbar(im2, ax=axes[1], label="FFT magnitude")

plt.suptitle("Spatial-Temporal Vibration Analysis", fontsize=14)
plt.tight_layout()
plt.show()

# Find nodes with highest vibration amplitude
node_total_energy = magnitudes.sum(axis=0)  # total FFT energy per node
top_nodes = np.argsort(node_total_energy)[-5:][::-1]
print("Nodes with highest vibration energy:")
for rank, node_idx in enumerate(top_nodes):
    print(f"  {rank+1}. Node {node_idx}: total FFT energy = {node_total_energy[node_idx]:.4f}")

---

## Summary & Bird Aero Application

This POC demonstrates a complete **ML-powered structural vibration analysis pipeline**, including **physics-aware training** that enforces boundary conditions, stress consistency, and smoothness constraints:

| Stage | What | Time (POC) | Time (Traditional FEA) |
|-------|------|-----------|------------------------|
| Training (baseline) | Learn deformation physics from FEA data | ~minutes (GPU) | N/A (offline) |
| Training (physics) | Add BC + stress + smoothness constraints | ~minutes (GPU) | N/A (offline) |
| Rollout | Predict 50-step deformation trajectory | ~seconds | ~hours per simulation |
| FFT | Extract dominant frequencies | ~milliseconds | ~seconds |
| Mode shapes | Identify structural resonances | ~milliseconds | ~minutes |

### Physics-Aware Training Impact

The physics-constrained model shows improved behavior at **boundary nodes** (fixed/clamped regions), where displacement should be near-zero. This is critical for structural analysis — incorrect boundary behavior propagates errors throughout the rollout and corrupts frequency analysis downstream.

### For Bird Aero's Products

This approach directly applies to Bird Aero's engineering workflow:

1. **AEROSHIELD / AMPS pods**: Predict how pod structures deform under flight vibration loads, identify potential fatigue points before physical testing

2. **SPREOS DIRCM**: Ensure optical alignment stability by verifying that vibration-induced deformations at the mounting interface stay within tolerance

3. **ASIO / OSCAR payloads**: Rapid design iteration — evaluate how geometry or material changes affect vibration characteristics without re-running expensive FEA simulations

### Next Steps for Production

- **Custom data**: Replace DeepMind dataset with Bird Aero's proprietary FEA simulations (COMSOL/ANSYS export → same graph format)
- **Longer rollouts**: Train with noise injection to reduce autoregressive drift over 100+ steps
- **Multi-physics**: Extend to coupled thermal-structural deformation
- **Real-time dashboard**: Serve trained model via API for interactive design exploration

In [ ]:
# Final summary
print("=" * 60)
print("Bird Aero Vibration POC — Results Summary")
print("=" * 60)
print(f"Model:            MeshGraphNet (hidden={train_config.hidden_dim}, layers={train_config.num_layers})")
print(f"Parameters:       {total_params:,}")
print(f"Training:         {train_config.epochs} epochs on {device}")
print(f"Test MSE:         {mean_mse:.6f} \u00b1 {std_mse:.6f}")
print(f"Rollout steps:    {NUM_ROLLOUT_STEPS}")
print(f"Dominant freqs:   {', '.join(f'{f:.4f}' for f, _ in top_freqs[:3])}")
print(f"")
print("Physics-aware training:")
print(f"  BC enforcement:  nodes types [1, 2, 3] (weight=1.0)")
print(f"  Stress loss:     weight=0.1")
print(f"  Smoothness:      weight=0.01")
if bc_mask.sum() > 0:
    reduction = (1 - np.mean(bc_mag_physics) / (np.mean(bc_mag_baseline) + 1e-8)) * 100
    print(f"  BC disp reduction: {reduction:.1f}%")
print(f"")
print(f"Outputs saved to: outputs/")
print(f"  - deformation.gif")
print(f"  - error_map.png")
print(f"  - frequency_spectrum.png")
print(f"  - mode_shapes.png")
print("=" * 60)